Entrenamiento usando el modelo *mistralai/Mistral-7B-Instruct-v0.3* dándole un enfoque discriminativo.

In [ ]:
# =================================================================
# PROTOCOLO DE RESCATE HUGGING FACE PARA COLAB PRO (L4)
# Objetivo: Evitar el bloqueo al 0% y solucionar Silent Socket Timeout
# =================================================================

import os
# Paso 1: Forzar variables de entorno antes de importar Hugging Face
os.environ["HF_HUB_DISABLE_XET"] = "1"        # Desactiva Xet (evita cuelgues en GCP)
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"  # Sube el timeout a 5 minutos por archivo
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"     # Mayor margen para negociar descargas

# Paso 2: Limpieza radical de semáforos de bloqueo residuales (.locks)
print("🧹 Limpiando bloqueos fantasma de intentos caídos anteriores...")
!rm -rf /root/.cache/huggingface/.locks
!rm -rf ~/.cache/huggingface/.locks

from google.colab import userdata
from huggingface_hub import login, snapshot_download
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Recuperamos tus credenciales seguras de los secretos de Colab
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

model_checkpoint = "mistralai/Mistral-7B-v0.3"

# Paso 3: Descarga ultra-estable secuencial de archivos grandes
print("🚀 Descargando archivos uno a uno de forma secuencial y robusta...")
try:
    # Usamos max_workers=1 para evitar que la red de Colab Pro
    # sature las conexiones de Hugging Face (lo que congelaba la descarga)
    local_dir = snapshot_download(
        repo_id=model_checkpoint,
        token=hf_token,
        max_workers=1,          # ¡ESTA ES LA CLAVE! Un hilo a la vez para estabilidad
        resume_download=True,   # Si la red tiene micro-cortes, continúa
        ignore_patterns=["*.msgpack", "*.h5"] # Solo descargamos Safetensors necesarios
    )
    print(f"✅ ¡Descarga completada con éxito en local: {local_dir}")
except Exception as e:
    print(f"❌ Error durante la descarga: {e}")
    print("Consejo: Si el error persiste, comprueba que has aceptado los términos en la web de HF.")

# Paso 4: Carga de componentes de forma local (100% offline)
print("⚙️ Inicializando el Tokenizador y el Modelo cuantizado...")
tokenizer = AutoTokenizer.from_pretrained(
    model_checkpoint,
    token=hf_token,
    padding_side="left"
)

# Cargamos los pesos de forma local sin hacer llamadas adicionales a internet
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    token=hf_token,
    local_files_only=True, # Fuerza a Hugging Face a no usar internet para instanciar
    device_map="auto"      # Mapea automáticamente a tu GPU Pro (L4)
)

print("🎉 ¡Modelo listo para entrenar! Ya puedes continuar con la celda de QLoRA.")


🧹 Limpiando bloqueos fantasma de intentos caídos anteriores...
🚀 Descargando archivos uno a uno de forma secuencial y robusta...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

❌ Error durante la descarga: (Request ID: 01KXGR0ZGT6RFNZJZ45T4V02VV)

403 Forbidden: None.
Cannot access content at: https://us.gcp.cdn.hf.co/xet-bridge-us/664dc156dba1a2aeb958dc90/c01f2a0f40c727edebca3ab0bad15d745f5cb0174ba383aad38e6974cb70b238?X-Xet-Cas-Uid=67f153fb098f94ab121ac6af&user_id=67f153fb098f94ab121ac6af&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27consolidated.safetensors%3B+filename%3D%22consolidated.safetensors%22%3B&Expires=1784050838&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjY0ZGMxNTZkYmExYTJhZWI5NThkYzkwL2MwMWYyYTBmNDBjNzI3ZWRlYmNhM2FiMGJhZDE1ZDc0NWY1Y2IwMTc0YmEzODNhYWQzOGU2OTc0Y2I3MGIyMzhcXD9YLVhldC1DYXMtVWlkPTY3ZjE1M2ZiMDk4Zjk0YWIxMjFhYzZhZiZ1c2VyX2lkPTY3ZjE1M2ZiMDk4Zjk0YWIxMjFhYzZhZiZyZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSoiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkVwb2NoVGltZSI6MTc4NDA1MDgzOH19fV19&Signature=MEUCIQDJTBuyXhiMvHCbkk6m2s4FZ4Fuqtrm%7EYanLlrS7d7boQIgZcDm3QaBxq0ZNov--3KaOVOBQI5fzda

tokenizer_config.json:   0%|          | 0.00/137k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

OSError: There was a specific connection error when trying to load mistralai/Mistral-7B-v0.3:
(Request ID: 01KXGR139C1CEDBANY02XAEAAD)

403 Forbidden: None.
Cannot access content at: https://us.gcp.cdn.hf.co/xet-bridge-us/664dc156dba1a2aeb958dc90/70de076fa18896073beef6149fbfc8ac2a287bc510c1a6f422ca4b8538b7a952?X-Xet-Cas-Uid=67f153fb098f94ab121ac6af&user_id=67f153fb098f94ab121ac6af&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27tokenizer.model%3B+filename%3D%22tokenizer.model%22%3B&Expires=1784050842&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjY0ZGMxNTZkYmExYTJhZWI5NThkYzkwLzcwZGUwNzZmYTE4ODk2MDczYmVlZjYxNDlmYmZjOGFjMmEyODdiYzUxMGMxYTZmNDIyY2E0Yjg1MzhiN2E5NTJcXD9YLVhldC1DYXMtVWlkPTY3ZjE1M2ZiMDk4Zjk0YWIxMjFhYzZhZiZ1c2VyX2lkPTY3ZjE1M2ZiMDk4Zjk0YWIxMjFhYzZhZiZyZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSoiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkVwb2NoVGltZSI6MTc4NDA1MDg0Mn19fV19&Signature=MEUCIGVtG1C88Ijeyv-Np82klxq6Izd0PCX5ZY

In [ ]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.4 MB/s eta 0:00:00


In [ ]:
import torch
import bitsandbytes as bnb
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset

In [ ]:
model_checkpoint = "mistralai/Mistral-7B-Instruct-v0.3"
n_labels = 2

1. Configuración de cuantización a 4 bits

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 # Acelera los cálculos matemáticos
)

print("Cargando tokenizador y modelo base")

Cargando tokenizador y modelo base


2. Tokenizador

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

3. Inicialización del Modelo Base Cuantizado (Con la cabeza matemática discriminativa)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=n_labels,
    quantization_config=bnb_config,
    device_map="auto"
)
model.config.pad_token_id = tokenizer.pad_token_id

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] MistralForSequenceClassification LOAD REPORT from: mistralai/Mistral-7B-Instruct-v0.3
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


4. Buscador dinámico de capas para LoRA

In [ ]:
def find_all_linear_names(model):
    cls = bnb.nn.Linear4bit
    lora_module_names = set()
    for name, module in model.named_modules():
        if isinstance(module, cls):
            names = name.split('.')
            lora_module_names.add(names[0] if len(names) == 1 else names[-1])
    if 'lm_head' in lora_module_names:
        lora_module_names.remove('lm_head')
    if 'score' in lora_module_names: # Excluimos la cabeza de clasificación de la cuantización normal
        lora_module_names.remove('score')
    return list(lora_module_names)

modules = find_all_linear_names(model)
print(f"Módulos objetivo encontrados para LoRA: {modules}")

Módulos objetivo encontrados para LoRA: ['up_proj', 'gate_proj', 'q_proj', 'down_proj', 'o_proj', 'k_proj', 'v_proj']


5. Configuración de LoRA

In [ ]:
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=modules,
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",      # Tarea: Clasificación de Secuencias (no CAUSAL_LM)
    modules_to_save=["score"] # IMPORTANTE: Descongelar la capa final matemática
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 41,951,232 || all params: 7,155,765,248 || trainable%: 0.5863


6. Preparación del dataset y partición

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ================================
# 1. Carga y preparación de datos
# ================================
import pandas as pd
from sklearn.model_selection import train_test_split
# from datasets import Dataset

print("Cargando el dataset maestro de entrenamiento...")
# Cargamos el TRAIN_MASTER que contiene el 80% de los datos (el test fijo ya está guardado aparte)
df_train_master = pd.read_csv("/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_training_3_1_master.csv")
test_df = pd.read_csv('/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_test_3_1.csv')

# Limpiamos el índice extra (si existe) y renombramos la columna
if "Unnamed: 0" in df_train_master.columns:
    df_train_master = df_train_master.drop(columns=["Unnamed: 0"])
df_train_master = df_train_master.rename(columns={"label_task_3_1_merged": "label"})
df_train_master["label"] = df_train_master["label"].astype(int)

if "Unnamed: 0" in test_df.columns:
    test_df = test_df.drop(columns=["Unnamed: 0"])
test_df = test_df.rename(columns={"label_task_3_1_merged": "label"})
test_df["label"] = test_df["label"].astype(int)

# División dinámica: 90% Train, 10% Validation (extraído solo del bloque maestro)
# Mantenemos random_state para que la validación sea estable entre pruebas del mismo modelo
train_df, val_df = train_test_split(df_train_master, test_size=0.10, stratify=df_train_master["label"], random_state=42)

print("Distribución fichero de entrenamiento (Train):")
print(train_df['label'].value_counts())

print("\nDistribución fichero de validación (Valid):")
print(val_df['label'].value_counts())

print("\nDistribución fichero de test (estático):")
print(test_df['label'].value_counts())

# train_dataset = Dataset.from_pandas(train_df)
# eval_dataset = Dataset.from_pandas(val_df)

Cargando el dataset maestro de entrenamiento...
Distribución fichero de entrenamiento (Train):
label
0    940
1    865
Name: count, dtype: int64

Distribución fichero de validación (Valid):
label
0    105
1     96
Name: count, dtype: int64

Distribución fichero de test (estático):
label
0    261
1    241
Name: count, dtype: int64


In [ ]:
def tokenize_data(example):
    # Asumimos que tu columna con el texto de los vídeos se llama "text"
    return tokenizer(example["text"], padding="max_length", truncation=True, max_length=128)

train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(val_df)

train_dataset.reset_format()
valid_dataset.reset_format()

columns_train = train_dataset.column_names
columns_valid = valid_dataset.column_names
columna_etiqueta = "label" if "label" in columns_train else "labels"

if columna_etiqueta in columns_train: columns_train.remove(columna_etiqueta)
if columna_etiqueta in columns_valid: columns_valid.remove(columna_etiqueta)

encoded_train_dataset = train_dataset.map(tokenize_data, batched=True, remove_columns=columns_train)
encoded_valid_dataset = valid_dataset.map(tokenize_data, batched=True, remove_columns=columns_valid)

Map:   0%|          | 0/1805 [00:00<?, ? examples/s]

Map:   0%|          | 0/201 [00:00<?, ? examples/s]

7. Hiperparámetros del entrenamiento

In [ ]:
training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/LLMs/Mistral_7B_QLoRA',
    num_train_epochs=5,
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=8,
    eval_strategy='epoch',      # CORREGIDO: Compatible con la última versión de Transformers
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    weight_decay=0.01,
    fp16=False,
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="none", # EVITAMOS el error de Weights & Biases
    logging_strategy='epoch'
)

8. Entrenamiento usando el trainer clásico (porque es el discriminativo)

In [ ]:
import sklearn as sk
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score, f1_score


In [ ]:
# Función para realizar distintas métricas en ejecución

def compute_metrics(eval_pred):

  ##############
  ## predictions son logits, que son tuplas de la forma [valor1, valor2]
  ## Por ejemplo [-1.5606991,  1.6122842] significa que ha predicho eso para un documento
  ## Eso es lo que pasa a la última capa del transformer (softmax si es binario)
  ## Por eso se utiliza el índice del valor máximo de la tupla, para decir que esa es la clase que predice

  ## label_ids = [0, 1, 1, 0, 1]  # Etiquetas reales
  ## predictions = [
  ##  [0.8, 0.2],  # Predicciones para la primera instancia
  ##  [0.3, 0.7],  # Predicciones para la segunda instancia
  ##  [0.1, 0.9],  # Predicciones para la tercera instancia
  ##  [0.9, 0.1],  # Predicciones para la cuarta instancia
  ##  [0.4, 0.6],  # Predicciones para la quinta instancia
  ##           ]

  ##############

  labels = eval_pred.label_ids
  preds = eval_pred.predictions.argmax(-1)

  # Compute precision, recall, F1-score, and support
  precision, recall, f1, _ = sk.metrics.precision_recall_fscore_support(labels, preds, average="macro")

  # Calculate F1-score for the minority class (label = 1)
  f1_minoritaria= f1_score(labels, preds, pos_label=1)

  # Calculate F1-score for the majority class (label = 0)
  f1_mayoritaria = f1_score(labels, preds, pos_label=0)

  # Calculate accuracy
  acc = sk.metrics.accuracy_score(labels, preds)

  # Calculate Area Under the Curve (AUC)
  AUC = roc_auc_score(labels, preds)

  # Calculate Precision-Recall Area Under the Curve (AUC)
  PREC_REC = average_precision_score(labels, preds)

  return {
      'accuracy': acc,
      'f1': f1,
      'precision': precision,
      'recall': recall,
      'AUC': AUC,
      'f1_minoritaria': f1_minoritaria,
      'f1_mayoritaria': f1_mayoritaria,
      'PREC_REC': PREC_REC
  }

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics, # ¡TU FUNCIÓN DE MÉTRICAS ORIGINAL!
    train_dataset=encoded_train_dataset,
    eval_dataset=encoded_valid_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("🚀 Iniciando entrenamiento de Mistral-7B...")
trainer.train()

🚀 Iniciando entrenamiento de Mistral-7B...


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Auc,F1 Minoritaria,F1 Mayoritaria,Prec Rec
1,4.099449,1.850634,0.477612,0.323232,0.238806,0.500000,0.500000,0.646465,0.000000,0.477612
2,4.683405,0.630633,0.621891,0.621431,0.626104,0.624702,0.624702,0.634615,0.608247,0.554388
3,1.339985,0.743665,0.681592,0.664790,0.706028,0.672917,0.672917,0.589744,0.739837,0.616117
4,0.690955,3.836748,0.691542,0.687450,0.693995,0.687798,0.687798,0.651685,0.723214,0.616392
5,0.158937,3.824752,0.701493,0.697532,0.704294,0.697768,0.697768,0.662921,0.732143,0.626280


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9,

TrainOutput(global_step=565, training_loss=2.1945460598025703, metrics={'train_runtime': 2460.2922, 'train_samples_per_second': 3.668, 'train_steps_per_second': 0.23, 'total_flos': 4.8667750170624e+16, 'train_loss': 2.1945460598025703, 'epoch': 5.0})

# Resultados contra fichero de Test

In [1]:
!pip install -U torchao accelerate transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 111.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 151.5 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, AutoModelForSequenceClassification
from peft import PeftModel

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Configuración

In [4]:
MODEL_ID_BASE = "mistralai/Mistral-7B-Instruct-v0.3" # Cambiar según el modelo
DIR_MODELO_QLORA = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/LLMs/Mistral_7B_QLoRA/checkpoint-565"

# Ruta al archivo de test estático
CSV_TEST_TEXT = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_test_3_1.csv"
CSV_SALIDA = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/LLMs/predicciones_mistral_test.csv"


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 2. Carga de datos de test

In [5]:
print("Cargando el dataset de test fijo...")
test_df = pd.read_csv(CSV_TEST_TEXT)

if "Unnamed: 0" in test_df.columns:
    test_df = test_df.drop(columns=["Unnamed: 0"])
test_df = test_df.rename(columns={"label_task_3_1_merged": "label"})
test_df["label"] = test_df["label"].astype(int)

# Guardamos la verdad absoluta
y_true = test_df["label"].tolist()

# Función para generar el prompt de evaluación (sin la respuesta)
def generate_test_prompt(data_point):
    return f"""
Clasifica el siguiente texto extraído de un vídeo de redes sociales en una de estas dos categorías: 'Misógino' o 'No misógino'. Devuelve ÚNICAMENTE la etiqueta correspondiente.
text: {data_point["text"]}
label: """.strip()

Cargando el dataset de test fijo...


## 3. Carga del modelo y tokenizador (PEFT/QLoRA)

In [6]:
print(f"Cargando Tokenizador de {MODEL_ID_BASE}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID_BASE)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Cargando Modelo Base (SequenceClassification) en 16-bits...")
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID_BASE,
    num_labels=2,            # Fundamental: Le decimos que es para 2 clases
    device_map="auto",
    torch_dtype=torch.bfloat16
)
base_model.config.pad_token_id = tokenizer.pad_token_id

print(f"Aplicando pesos QLoRA (incluyendo la capa de 'score') desde {DIR_MODELO_QLORA}...")
# Esto cargará la cabeza clasificadora gracias a tu "modules_to_save=['score']"
model = PeftModel.from_pretrained(base_model, DIR_MODELO_QLORA)
model.eval()

Cargando Tokenizador de mistralai/Mistral-7B-Instruct-v0.3...


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  587kB            

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] MistralForSequenceClassification LOAD REPORT from: mistralai/Mistral-7B-Instruct-v0.3
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Aplicando pesos QLoRA (incluyendo la capa de 'score') desde /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/LLMs/Mistral_7B_QLoRA/checkpoint-565...


PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): MistralForSequenceClassification(
      (model): MistralModel(
        (embed_tokens): Embedding(32768, 4096)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )


## 4. Inferencia y parseo de las respuestas

In [7]:
print("Iniciando inferencia (extracción de probabilidades)...")
predicciones_csv = []
y_pred = []

for index, row in tqdm(test_df.iterrows(), total=len(test_df)):
    # Usamos el texto puro, tal cual lo hiciste en el entrenamiento
    texto = str(row["text"])
    id_video = row["id_EXIST"]

    # Tokenizamos
    inputs = tokenizer(texto, return_tensors="pt", padding="max_length", truncation=True, max_length=128).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

        # Obtenemos los logits y aplicamos Softmax para sacar porcentajes (0 a 1)
        logits = outputs.logits
        probabilidades = F.softmax(logits, dim=-1)[0]

        # probabilidad de la clase 1 (Misógino)
        prob_misogino = probabilidades[1].item()

    # Decisión binaria (Umbral estándar > 0.5)
    pred_binaria = 1 if prob_misogino > 0.5 else 0
    y_pred.append(pred_binaria)

    # Guardamos los datos para el Ensemble
    predicciones_csv.append({
        "id_EXIST": id_video,
        "prob_misogino": prob_misogino,
        "prediccion_binaria": pred_binaria,
        "label_real": row["label"]
    })

Iniciando inferencia (extracción de probabilidades)...


100%|██████████| 502/502 [00:49<00:00, 10.07it/s]


## 5. Evaluación y métricas

In [8]:
print("\n" + "="*50)
print(f"🏆 RESULTADOS REALES EN TEST FIJO: Mistral (Sequence Classification)")
print("="*50)

f1 = f1_score(y_true, y_pred, average="macro")
acc = accuracy_score(y_true, y_pred)

print(f"\nF1-Score (Macro): {f1:.4f}")
print(f"Accuracy: {acc:.4f}")

print("\nMatriz de Confusión:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["No Misógino", "Misógino"]))


🏆 RESULTADOS REALES EN TEST FIJO: Mistral (Sequence Classification)

F1-Score (Macro): 0.7141
Accuracy: 0.7171

Matriz de Confusión:
[[206  55]
 [ 87 154]]

Classification Report:
              precision    recall  f1-score   support

 No Misógino       0.70      0.79      0.74       261
    Misógino       0.74      0.64      0.68       241

    accuracy                           0.72       502
   macro avg       0.72      0.71      0.71       502
weighted avg       0.72      0.72      0.72       502



## 6. Guardar CSV para ensemble

In [9]:
df_salida = pd.DataFrame(predicciones_csv)
df_salida.to_csv(CSV_SALIDA, index=False)
print(f"\n✅ ¡CSV para el Ensemble guardado correctamente en: {CSV_SALIDA}!")


✅ ¡CSV para el Ensemble guardado correctamente en: /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/LLMs/predicciones_mistral_test.csv!
